# FakeQubexExperiment calibration workflow

This notebook uses `FakeQubexExperiment` through QUBEX `Experiment`-like calibration methods. The calibration details live in the fake experiment API; the notebook keeps the same calling style as the hardware notebooks.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector
from qxsimulator import QuantumSimulator

from qiskit_qubex_provider import (
    FakeQubexExperiment,
    QubexProvider,
    build_qxsimulator_system,
    filter_pulse_schedule_for_simulation,
    materialize_pulse_schedule_for_simulation,
)


## Create a fake experiment

Instantiate the fake experiment with explicit model parameters. The coherence times are intentionally short; they are written to `device_topology` and used by qxsimulator helpers. The local Qiskit sampling backend created by `from_device_topology(...)` does not apply T1/T2 noise by itself.

In [ ]:
exp = FakeQubexExperiment(
    name="fake-qubex-bad-coherence-demo",
    device_id="fake-qubex-bad-coherence-demo",
    qubit_labels=("Q00", "Q01"),
    qubit_frequencies=(7.157231, 8.032295),  # GHz
    qubit_anharmonicities=(-0.393715, -0.487412),  # GHz
    coupling_strength=0.005,  # GHz
    qubit_lifetimes=(
        (12.0, 8.0),  # Q00: T1/T2 in microseconds, intentionally mediocre
        (14.0, 9.0),  # Q01: T1/T2 in microseconds, intentionally mediocre
    ),
    hpi_duration=24.0,  # ns; initial pulse design parameter
    pi_duration=24.0,  # ns; initial pulse design parameter
    readout_duration=1000.0,  # ns; readout pulse design parameter
)
control, target = exp.qubit_labels

print(exp.qubit_labels)
print(exp.device_topology()["qubits"])
print(exp.device_topology()["couplings"])


## Calibrate single-qubit DRAG pulses

This mirrors the hardware calibration entry point. The result stores the calibrated pulse parameters and repeat-sequence data.

In [ ]:
result_drag_hpi = exp.calibrate_drag_hpi_pulse(
    exp.qubit_labels,
    repetitions=20,
)
result_drag_pi = exp.calibrate_drag_pi_pulse(exp.qubit_labels)

{
    qubit: {
        "hpi_duration": result_drag_hpi[qubit]["duration"],
        "hpi_amplitude": result_drag_hpi[qubit]["amplitude"],
        "hpi_beta": result_drag_hpi[qubit]["beta"],
        "pi_duration": result_drag_pi[qubit]["duration"],
        "pi_amplitude": result_drag_pi[qubit]["amplitude"],
        "pi_beta": result_drag_pi[qubit]["beta"],
    }
    for qubit in exp.qubit_labels
}

In [ ]:
for qubit in exp.qubit_labels:
    result_drag_hpi.get_figure(qubit).show()

## Obtain CR parameters

`obtain_cr_params` performs the simulated Hamiltonian tomography, cancellation calibration, and echoed ZX90 amplitude calibration.

In [ ]:
result_cr = exp.obtain_cr_params(
    control,
    target,
    tomography_duration=400,
    tomography_samples=80,
)

result_cr["cr_param"]

In [ ]:
result_cr.get_figure("amplitude").show()

## CR Hamiltonian tomography terms

Inspect the estimated CR Hamiltonian coefficients before relying on the pulse in algorithm simulations. Large residual `IX/IY/ZY/ZZ` terms indicate frame or cancellation errors that are not visible in a simple repeat sequence.


In [ ]:
import pandas as pd

cr_hamiltonian_rows = []
for index, tomography in enumerate(result_cr["tomography"]):
    row = {"tomography_index": index}
    row.update({term: float(value) for term, value in tomography["coeffs"].items()})
    row["abs_xt"] = abs(tomography["xt_rotation"])
    row["abs_cr"] = abs(tomography["cr_rotation"])
    row["zx90_duration"] = float(tomography["zx90_duration"])
    cr_hamiltonian_rows.append(row)

cr_hamiltonian_df = pd.DataFrame(cr_hamiltonian_rows)
cr_hamiltonian_df


In [ ]:
import plotly.graph_objects as go

fig = go.Figure()
terms = ["IX", "IY", "IZ", "ZX", "ZY", "ZZ"]
for _, row in cr_hamiltonian_df.iterrows():
    fig.add_trace(go.Bar(
        x=terms,
        y=[row[term] for term in terms],
        name=f"tomography {int(row['tomography_index'])}",
    ))
fig.update_layout(
    title="CR Hamiltonian tomography coefficients",
    xaxis_title="Hamiltonian term",
    yaxis_title="coefficient [1/ns]",
    barmode="group",
)
fig.show()


## Fine tune ZX90

`obtain_cr_params` gives a coarse echoed ZX90. The fake experiment then performs a lightweight `calibrate_zx90` amplitude fine tune using qxsimulator before using the pulse in longer circuits.

In [ ]:
result_zx90 = exp.calibrate_zx90(
    control,
    target,
    n_repetitions=8,
    n_points=21,
)

result_zx90["cr_param"]


In [ ]:
result_zx90.get_figure("amplitude").show()

## Check the calibrated ZX90 pulse

This is the same shape as the experiment notebook: build `zx90`, then call `repeat_sequence`.

In [ ]:
zx90 = exp.zx90(control, target)
repeat_result = exp.repeat_sequence(
    zx90,
    repetitions=20,
    initial_state={control: "0"},
)

repeat_result["dataframe"]

In [ ]:
repeat_result.get_figure().show()

## Pulse tomography and Bell checks

These cells mirror the QUBEX experiment CR calibration notebook checks after building the calibrated ZX90 pulse.

In [ ]:
pulse_tomography_0 = exp.pulse_tomography(
    zx90,
    initial_state={control: "0"},
    n_samples=80,
)

pulse_tomography_0.keys()


In [ ]:
pulse_tomography_1 = exp.pulse_tomography(
    zx90,
    initial_state={control: "1"},
    n_samples=80,
)

pulse_tomography_1.keys()


In [ ]:
result_bell = exp.measure_bell_state(control, target)
result_bell["mitigated"]


In [ ]:
result_bell_tomography = exp.bell_state_tomography(control, target)
result_bell_tomography["fidelity"]


In [ ]:
result_zx90_fidelity = exp.zx90_gate_fidelity(control, target)
result_cx_fidelity = exp.cx_gate_fidelity(control, target)
result_cx_frame = exp.calibrate_cx_frame(control, target)

{
    "zx90_fidelity": result_zx90_fidelity["fidelity"],
    "cx_fidelity": result_cx_fidelity["fidelity"],
    "cx_dressed_fidelity": result_cx_frame["dressed_fidelity"],
    "cx_post_control_z": result_cx_frame["post_control_z"],
    "cx_post_target_z": result_cx_frame["post_target_z"],
}


## Pulse-schedule randomized benchmarking

This diagnostic is intentionally disabled by default because pulse-level 2Q RB is slow in qxsimulator. Re-enable it when checking whether QUBEX-style Clifford RB agrees with the calibrated pulse convention.


```python
cr_label = f"{control}-{target}"
rb_n_cliffords = [0, 1, 2, 4, 8, 16]
rb_seeds = list(range(1, 9))

rb_result = exp.randomized_benchmarking(
    cr_label,
    n_cliffords_range=rb_n_cliffords,
    n_trials=len(rb_seeds),
    seeds=rb_seeds,
    plot=True,
)

irb_zx90_result = exp.interleaved_randomized_benchmarking(
    cr_label,
    interleaved_clifford="ZX90",
    n_cliffords_range=rb_n_cliffords,
    n_trials=len(rb_seeds),
    seeds=rb_seeds,
    plot=True,
)

{
    "rb_avg_gate_fidelity": rb_result[cr_label].get("avg_gate_fidelity"),
    "rb_avg_gate_error": rb_result[cr_label].get("avg_gate_error"),
    "irb_zx90_avg_gate_fidelity": irb_zx90_result[cr_label].get("avg_gate_fidelity"),
    "irb_zx90_avg_gate_error": irb_zx90_result[cr_label].get("avg_gate_error"),
    "rb": rb_result[cr_label]["dataframe"],
    "irb_zx90": irb_zx90_result[cr_label]["dataframe"],
}
```


```python
rb_result.get_figure(cr_label).show()
irb_zx90_result.get_figure(cr_label).show()
```


## Build state classifiers

The fake experiment generates synthetic IQ data and fits the same GMM classifier used by the QUBEX measurement workflow.

In [ ]:
binary_distributions = exp.measure_state_distribution(
    exp.qubit_labels,
    n_states=2,
    n_shots=10_000,
)

result_classifier = exp.build_classifier(
    exp.qubit_labels,
    n_states=2,
    n_shots=10_000,
)

result_classifier["average_readout_fidelity"]

In [ ]:
for qubit in exp.qubit_labels:
    result_classifier.get_figure(f"{qubit}:0").show()
    result_classifier.get_figure(f"{qubit}:1").show()

## Export calibrated topology and build the provider

The CR calibration updates the fake experiment durations, and classifier calibration adds readout assignment errors. The calibrated topology is used both for the Qiskit provider target and for the qxsimulator model.

In [ ]:
topology = exp.device_topology()
provider = QubexProvider.from_experiment(
    exp,
    device_topology=topology,
    refresh_instruction_durations=False,
)
backend = provider.get_backend()
provider_no_cr_frame_sync = QubexProvider.from_experiment(
    exp,
    device_topology=topology,
    refresh_instruction_durations=False,
    sync_cr_channel_frames=False,
)
backend_no_cr_frame_sync = provider_no_cr_frame_sync.get_backend()
qx_system = build_qxsimulator_system(
    topology,
    qubit_labels=exp.qubit_labels,
    qubit_dimension=3,
    include_decoherence=True,
)
qx_system_no_decoherence = build_qxsimulator_system(
    topology,
    qubit_labels=exp.qubit_labels,
    qubit_dimension=3,
    include_decoherence=False,
)
pulse_simulator = QuantumSimulator(qx_system)
pulse_simulator_no_decoherence = QuantumSimulator(qx_system_no_decoherence)

print("provider qubits:", backend.num_qubits)
print("CR frame sync enabled:", backend.executor._sync_cr_channel_frames_enabled)
print("CR frame sync disabled backend:", backend_no_cr_frame_sync.executor._sync_cr_channel_frames_enabled)
print("sx duration [s]:", backend.target["sx"][(0,)].duration)
print("cx duration [s]:", backend.target["cx"][(0, 1)].duration)
print("qxsimulator objects:", qx_system.object_labels)
print("coupling [GHz]:", qx_system.get_coupling(tuple(exp.qubit_labels)).strength)
print("lifetimes [us]:", [q["qubit_lifetime"] for q in topology["qubits"]])

exp.write_device_topology(
    "fake-calibrated-device-topology.json",
    image_path="fake-calibrated-device-topology.svg",
)


## Primitive pulse sanity checks

Before running a long Heisenberg sequence, check the calibrated pulse schedules for simple circuits. If these are already far from the gate-level reference, the Heisenberg mismatch is calibration/pulse-model error rather than coherence time.

In [ ]:
def simulate_pulse_circuit_probability(circuit: QuantumCircuit, *, simulator: QuantumSimulator) -> dict[str, float]:
    scheduled = transpile(
        circuit,
        backend,
        scheduling_method="alap",
        optimization_level=1,
        seed_transpiler=7,
    )
    schedule = backend.validate(scheduled)[0]
    schedule = filter_pulse_schedule_for_simulation(
        schedule,
        labels=[*exp.qubit_labels, f"{control}-{target}"],
    )
    if not schedule.labels:
        return Statevector.from_instruction(circuit).probabilities_dict()
    schedule = materialize_pulse_schedule_for_simulation(schedule)
    result = simulator.mesolve(schedule, n_samples=2)
    populations = np.real(result.final_state.diag())
    return {
        label: float(populations[index])
        for index, label in enumerate(result.system.basis_labels)
        if len(label) == 2 and set(label) <= {"0", "1"}
    }


def circuit_probabilities(circuit: QuantumCircuit) -> dict[str, float]:
    return Statevector.from_instruction(circuit).probabilities_dict()


primitive_circuits = {}
primitive_circuits["cx |00>"] = QuantumCircuit(2)
primitive_circuits["cx |00>"].cx(0, 1)
primitive_circuits["bell"] = QuantumCircuit(2)
primitive_circuits["bell"].h(0)
primitive_circuits["bell"].cx(0, 1)
primitive_circuits["cx-cx |00>"] = QuantumCircuit(2)
primitive_circuits["cx-cx |00>"].cx(0, 1)
primitive_circuits["cx-cx |00>"].cx(0, 1)
primitive_circuits["cx-rz(0.2)-cx |00>"] = QuantumCircuit(2)
primitive_circuits["cx-rz(0.2)-cx |00>"].cx(0, 1)
primitive_circuits["cx-rz(0.2)-cx |00>"].rz(0.2, 1)
primitive_circuits["cx-rz(0.2)-cx |00>"].cx(0, 1)

primitive_rows = []
for name, circuit in primitive_circuits.items():
    exact = circuit_probabilities(circuit)
    no_decoh = simulate_pulse_circuit_probability(
        circuit,
        simulator=pulse_simulator_no_decoherence,
    )
    decoh = simulate_pulse_circuit_probability(circuit, simulator=pulse_simulator)
    primitive_rows.append(
        {
            "circuit": name,
            "exact_P00": exact.get("00", 0.0),
            "exact_P11": exact.get("11", 0.0),
            "pulse_no_decoh_P00": no_decoh.get("00", 0.0),
            "pulse_no_decoh_P11": no_decoh.get("11", 0.0),
            "pulse_T1T2_P00": decoh.get("00", 0.0),
            "pulse_T1T2_P11": decoh.get("11", 0.0),
        }
    )
primitive_rows


## Simulation/hardware divergence probes

Use these probes when the real device behaves better than the pulse simulator. They keep the gate-level circuit fixed and expose schedule metadata, frame shifts, waveform norms, and the pulse-level unitary population for the same primitive circuits.


In [ ]:
def schedule_for_probe(
    circuit: QuantumCircuit,
    *,
    schedule_backend=backend,
    scheduling_method: str = "alap",
):
    scheduled = transpile(
        circuit,
        schedule_backend,
        scheduling_method=scheduling_method,
        optimization_level=1,
        seed_transpiler=7,
    )
    schedule = schedule_backend.validate(scheduled)[0]
    return filter_pulse_schedule_for_simulation(
        schedule,
        labels=[*exp.qubit_labels, f"{control}-{target}"],
    )


def materialized_probe_schedule(circuit: QuantumCircuit, *, schedule_backend=backend):
    schedule = schedule_for_probe(circuit, schedule_backend=schedule_backend)
    if not schedule.labels:
        return schedule
    return materialize_pulse_schedule_for_simulation(schedule)


def _target_metadata(schedule, label: str) -> dict[str, object]:
    try:
        target_obj = schedule.get_target(label)
    except Exception:
        return {"target_object": None, "frequency_GHz": None}
    obj = getattr(target_obj, "object", target_obj)
    return {
        "target_object": getattr(obj, "label", repr(obj)),
        "frequency_GHz": getattr(target_obj, "frequency", None),
    }


def _channel_waveform_norm(schedule, label: str) -> float | None:
    try:
        values = np.asarray(schedule[label].get_sequence().values, dtype=complex)
    except Exception:
        return None
    return float(np.linalg.norm(values))


def schedule_probe_dataframe(circuits: dict[str, QuantumCircuit], *, schedule_backend=backend) -> pd.DataFrame:
    rows = []
    for circuit_name, circuit in circuits.items():
        schedule = schedule_for_probe(circuit, schedule_backend=schedule_backend)
        materialized = materialize_pulse_schedule_for_simulation(schedule) if schedule.labels else schedule
        for label in schedule.labels:
            metadata = _target_metadata(schedule, label)
            rows.append(
                {
                    "circuit": circuit_name,
                    "label": label,
                    "target_object": metadata["target_object"],
                    "frequency_GHz": metadata["frequency_GHz"],
                    "final_frame_shift": float(schedule.get_final_frame_shift(label)),
                    "materialized_final_frame_shift": float(materialized.get_final_frame_shift(label)),
                    "waveform_norm": _channel_waveform_norm(materialized, label),
                }
            )
    return pd.DataFrame(rows)


def pulse_p00_from_propagator(circuit: QuantumCircuit, *, schedule_backend=backend) -> float:
    schedule = materialized_probe_schedule(circuit, schedule_backend=schedule_backend)
    if not schedule.labels:
        return float(Statevector.from_instruction(circuit).probabilities_dict().get("00", 0.0))
    unitary = pulse_simulator_no_decoherence.propagator(schedule).full()
    initial = np.zeros(unitary.shape[1], dtype=complex)
    initial[qx_system_no_decoherence.basis_labels.index("00")] = 1.0
    final = unitary @ initial
    return float(abs(final[qx_system_no_decoherence.basis_labels.index("00")]) ** 2)


probe_rows = []
for name, circuit in primitive_circuits.items():
    exact = float(Statevector.from_instruction(circuit).probabilities_dict().get("00", 0.0))
    probe_rows.append(
        {
            "circuit": name,
            "exact_P00": exact,
            "pulse_propagator_P00": pulse_p00_from_propagator(circuit),
            "delta": pulse_p00_from_propagator(circuit) - exact,
        }
    )

probe_population_df = pd.DataFrame(probe_rows)
probe_population_df


In [ ]:
schedule_probe_df = schedule_probe_dataframe(primitive_circuits)
schedule_probe_df


In [ ]:
cx_schedule = materialized_probe_schedule(primitive_circuits["cx |00>"])
cnot_schedule = materialize_pulse_schedule_for_simulation(exp.cnot(control, target))
{
    "cx_labels": list(cx_schedule.labels),
    "cnot_labels": list(cnot_schedule.labels),
    "cx_final_frames": {label: cx_schedule.get_final_frame_shift(label) for label in cx_schedule.labels},
    "cnot_final_frames": {label: cnot_schedule.get_final_frame_shift(label) for label in cnot_schedule.labels},
    "waveform_norm_diffs": {
        label: abs(
            (_channel_waveform_norm(cx_schedule, label) or 0.0)
            - (_channel_waveform_norm(cnot_schedule, label) or 0.0)
        )
        for label in cx_schedule.labels
    },
}


## 2-qubit Heisenberg pulse-simulation comparison

This comparison uses the calibrated fake provider to build QUBEX pulse schedules, then runs those schedules with `qxsimulator.QuantumSimulator`. The statevector curve is the gate-level reference. The ALAP/ASAP curves include pulse calibration, the calibrated `device_topology`, and the intentionally short T1/T2 values.

In [ ]:
NUM_QUBITS = 2
N_STEPS = 2
REFERENCE_J = 1.0
TOTAL_TIMES = np.linspace(0, 2 * np.pi, 41)
TOTAL_TIMES_IDEAL = np.linspace(0, 2 * np.pi, 201)
PULSE_SIMULATION_SAMPLES = 2


def create_half_ghz_state(qc: QuantumCircuit) -> None:
    qc.h(0)


def uncompute_half_ghz_state(qc: QuantumCircuit) -> None:
    qc.h(0)


def add_xz_pauli_rotation(qc: QuantumCircuit, q0: int, q1: int, angle: float) -> None:
    qc.h(q0)
    qc.rzz(angle, q0, q1)
    qc.h(q0)


def add_heisenberg_interactions(
    qc: QuantumCircuit,
    edge: tuple[int, int],
    coupling: float,
    evolution_time: float,
    *,
    cx_post_control_z: float = 0.0,
    cx_post_target_z: float = 0.0,
) -> None:
    control_index, target_index = edge
    theta = float(coupling) * evolution_time
    qc.cx(control_index, target_index)
    if cx_post_control_z:
        qc.rz(cx_post_control_z, control_index)
    if cx_post_target_z:
        qc.rz(cx_post_target_z, target_index)
    qc.rx(2.0 * theta, control_index)
    qc.rz(2.0 * theta, target_index)
    add_xz_pauli_rotation(qc, control_index, target_index, -2.0 * theta)
    qc.cx(control_index, target_index)
    if cx_post_control_z:
        qc.rz(cx_post_control_z, control_index)
    if cx_post_target_z:
        qc.rz(cx_post_target_z, target_index)


def heisenberg_2q_circuit(
    total_time: float,
    *,
    n_steps: int = N_STEPS,
    cx_post_control_z: float = 0.0,
    cx_post_target_z: float = 0.0,
) -> QuantumCircuit:
    qc = QuantumCircuit(NUM_QUBITS)
    create_half_ghz_state(qc)
    step_time = float(total_time) / n_steps
    for _ in range(n_steps):
        add_heisenberg_interactions(
            qc,
            (0, 1),
            REFERENCE_J,
            step_time,
            cx_post_control_z=cx_post_control_z,
            cx_post_target_z=cx_post_target_z,
        )
    uncompute_half_ghz_state(qc)
    return qc


def dressed_heisenberg_2q_circuit(total_time: float, *, n_steps: int = N_STEPS) -> QuantumCircuit:
    return heisenberg_2q_circuit(
        total_time,
        n_steps=n_steps,
        cx_post_control_z=result_cx_frame["post_control_z"],
        cx_post_target_z=result_cx_frame["post_target_z"],
    )


def exact_probability_zero(circuit: QuantumCircuit) -> float:
    probabilities = Statevector.from_instruction(circuit).probabilities_dict()
    return float(probabilities.get("0" * NUM_QUBITS, 0.0))


def compile_with_policy(total_time: float, scheduling_method: str) -> QuantumCircuit:
    return transpile(
        heisenberg_2q_circuit(total_time),
        backend,
        scheduling_method=scheduling_method,
        optimization_level=1,
        seed_transpiler=7,
    )


def scheduled_duration_ns(circuit: QuantumCircuit) -> float:
    duration_dt = circuit._duration if getattr(circuit, "_duration", None) is not None else circuit.duration
    return float(duration_dt * backend.dt / 1e-9)


def pulse_schedule_for_simulation(circuit: QuantumCircuit):
    schedule = backend.validate(circuit)[0]
    schedule = filter_pulse_schedule_for_simulation(
        schedule,
        labels=[*exp.qubit_labels, f"{control}-{target}"],
    )
    if not schedule.labels:
        return None
    return materialize_pulse_schedule_for_simulation(schedule)


def pulse_probability_zero(circuit: QuantumCircuit, *, simulator: QuantumSimulator, fallback: float) -> float:
    schedule = pulse_schedule_for_simulation(circuit)
    if schedule is None:
        return fallback
    result = simulator.mesolve(
        schedule,
        n_samples=PULSE_SIMULATION_SAMPLES,
    )
    populations = np.real(result.final_state.diag())
    return float(populations[result.system.basis_labels.index("00")])


heisenberg_2q_circuit(np.pi / 2).draw("text")


## CR frame sync sensitivity

This diagnostic compares the same scheduled circuit with provider CR-channel frame synchronization enabled and disabled.

In [ ]:
def pulse_probability_zero_with_backend(
    circuit: QuantumCircuit,
    *,
    schedule_backend,
    simulator: QuantumSimulator,
    fallback: float,
) -> float:
    schedule = schedule_backend.validate(circuit)[0]
    schedule = filter_pulse_schedule_for_simulation(
        schedule,
        labels=[*exp.qubit_labels, f"{control}-{target}"],
    )
    if not schedule.labels:
        return fallback
    schedule = materialize_pulse_schedule_for_simulation(schedule)
    result = simulator.mesolve(schedule, n_samples=PULSE_SIMULATION_SAMPLES)
    populations = np.real(result.final_state.diag())
    return float(populations[result.system.basis_labels.index("00")])


sync_sample_time = float(TOTAL_TIMES[1])
sync_sample = compile_with_policy(sync_sample_time, "asap")
sync_reference = exact_probability_zero(heisenberg_2q_circuit(sync_sample_time))
{
    "total_time": sync_sample_time,
    "statevector": sync_reference,
    "pulse_sync_on": pulse_probability_zero_with_backend(
        sync_sample,
        schedule_backend=backend,
        simulator=pulse_simulator_no_decoherence,
        fallback=sync_reference,
    ),
    "pulse_sync_off": pulse_probability_zero_with_backend(
        sync_sample,
        schedule_backend=backend_no_cr_frame_sync,
        simulator=pulse_simulator_no_decoherence,
        fallback=sync_reference,
    ),
}


In [ ]:
ideal_curve = [
    exact_probability_zero(heisenberg_2q_circuit(float(total_time)))
    for total_time in TOTAL_TIMES_IDEAL
]
dressed_ideal_curve = [
    exact_probability_zero(dressed_heisenberg_2q_circuit(float(total_time)))
    for total_time in TOTAL_TIMES_IDEAL
]
comparison_rows = []
for total_time in TOTAL_TIMES:
    row = {
        "total_time": float(total_time),
        "statevector": exact_probability_zero(heisenberg_2q_circuit(float(total_time))),
        "dressed_statevector": exact_probability_zero(dressed_heisenberg_2q_circuit(float(total_time))),
    }
    for policy in ("alap", "asap"):
        scheduled = compile_with_policy(float(total_time), policy)
        row[f"{policy}_pulse_p00_nodecoh"] = pulse_probability_zero(
            scheduled,
            simulator=pulse_simulator_no_decoherence,
            fallback=row["statevector"],
        )
        row[f"{policy}_pulse_p00"] = pulse_probability_zero(
            scheduled,
            simulator=pulse_simulator,
            fallback=row["statevector"],
        )
        row[f"{policy}_duration_ns"] = scheduled_duration_ns(scheduled)
    comparison_rows.append(row)

comparison_rows[:3]


In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=TOTAL_TIMES_IDEAL,
    y=ideal_curve,
    name="gate-level statevector reference",
    line=dict(color="#475569", dash="dot"),
))
fig.add_trace(go.Scatter(
    x=TOTAL_TIMES_IDEAL,
    y=dressed_ideal_curve,
    name="dressed CX statevector reference",
    line=dict(color="#16a34a", dash="dot"),
))
for policy, color in (("alap", "#2563eb"), ("asap", "#dc2626")):
    fig.add_trace(go.Scatter(
        x=[row["total_time"] for row in comparison_rows],
        y=[row[f"{policy}_pulse_p00_nodecoh"] for row in comparison_rows],
        mode="lines+markers",
        name=f"{policy.upper()} pulse schedule, no decoherence",
        line=dict(color=color, dash="dash"),
    ))
    fig.add_trace(go.Scatter(
        x=[row["total_time"] for row in comparison_rows],
        y=[row[f"{policy}_pulse_p00"] for row in comparison_rows],
        mode="lines+markers",
        name=f"{policy.upper()} pulse schedule, T1/T2",
        line=dict(color=color),
    ))
fig.update_layout(
    title="2q Heisenberg pulse simulation with calibrated FakeQubexExperiment",
    xaxis=dict(title="total evolution time"),
    yaxis=dict(title="population P(|00>)", range=[-0.05, 1.05]),
    width=880,
    height=500,
)
fig.show()


In [ ]:
fig = go.Figure()
for policy, color in (("alap", "#2563eb"), ("asap", "#dc2626")):
    fig.add_trace(go.Scatter(
        x=[row["total_time"] for row in comparison_rows],
        y=[row[f"{policy}_duration_ns"] for row in comparison_rows],
        mode="lines+markers",
        name=f"{policy.upper()} scheduled duration",
        line=dict(color=color),
    ))
fig.update_layout(
    title="Scheduled duration from calibrated device topology",
    xaxis=dict(title="total evolution time"),
    yaxis=dict(title="scheduled duration [ns]"),
    width=760,
    height=430,
)
fig.show()
